# VoiceCraft Hebrew Finetuning

This notebook finetunes VoiceCraft (English TTS model) on Hebrew speech data.

**Steps:**
1. Install dependencies & clone VoiceCraft
2. Build Hebrew phonemizer (espeak doesn't support Hebrew)
3. Prepare Hebrew training data from Google FLEURS
4. Finetune the pretrained model on Hebrew
5. Test speech editing in Hebrew

**Requirements:** Kaggle with GPU T4 x2, Internet ON

## Step 1: Install Dependencies & Clone VoiceCraft

In [ ]:
import os, sys

# Clone VoiceCraft
if not os.path.exists('/kaggle/working/VoiceCraft'):
    !git clone https://github.com/jasonppy/VoiceCraft.git /kaggle/working/VoiceCraft

os.chdir('/kaggle/working/VoiceCraft')
sys.path.insert(0, '/kaggle/working/VoiceCraft')
sys.path.insert(0, '/kaggle/working/VoiceCraft/src')

# Install packages
!pip install -q encodec soundfile librosa 2>&1 | tail -1
!pip install -q datasets==2.16.0 2>&1 | tail -1
!pip install -q av einops flashy julius num2words sentencepiece hydra-core hydra_colorlog 2>&1 | tail -1
!pip install --no-dependencies -q git+https://git@github.com/facebookresearch/audiocraft#egg=audiocraft 2>&1 | tail -3

# Fix torchaudio version to match PyTorch
!pip install -q torchaudio==2.8.0+cu126 --index-url https://download.pytorch.org/whl/cu126 2>&1 | tail -3

print("\n✅ Packages installed")

### Restart kernel after installs\n\nRun the cell below, then continue from Step 2 after kernel restarts.

In [ ]:
import os
os._exit(0)

## Step 2: Setup Model & Hebrew Phonemizer\n\nRun this cell after kernel restart. It does everything:\n- Patches VoiceCraft to remove broken dependencies\n- Loads the pretrained model\n- Creates our custom Hebrew phonemizer

In [ ]:
import sys, os, gc, json, time, random, argparse, logging, pickle
import numpy as np
import torch
import torch.nn as nn
import torchaudio

os.chdir('/kaggle/working/VoiceCraft')
sys.path.insert(0, '/kaggle/working/VoiceCraft')
device = torch.device("cuda")
logging.basicConfig(level=logging.INFO)
print(f"GPU: {torch.cuda.get_device_name(0)}")

# --- Patch VoiceCraft: replace broken torchmetrics import ---
# VoiceCraft imports MulticlassAccuracy from torchmetrics which is incompatible
# with Kaggle's Python 3.12. We replace it with a dummy since it's only used
# for logging accuracy during training, not for the actual model.
os.chdir('/kaggle/working/VoiceCraft')
with open('models/voicecraft.py', 'r') as f:
    code = f.read()
if 'from torchmetrics' in code:
    code = code.replace(
        'from torchmetrics.classification import MulticlassAccuracy',
        'import torch.nn as nn\n'
        'class MulticlassAccuracy(nn.Module):\n'
        '    def __init__(self, *args, **kwargs): super().__init__()\n'
        '    def forward(self, *args, **kwargs): return torch.tensor(0.0)\n'
        '    def update(self, *args, **kwargs): pass\n'
        '    def compute(self, *args, **kwargs): return torch.tensor(0.0)\n'
        '    def reset(self, *args, **kwargs): pass'
    )
    with open('models/voicecraft.py', 'w') as f:
        f.write(code)
    print("✅ Patched voicecraft.py (removed torchmetrics dependency)")

from models import voicecraft
print("✅ VoiceCraft imported")

# --- Load pretrained model ---
import requests
voicecraft_name = "giga330M.pth"
ckpt_fn = f"./pretrained_models/{voicecraft_name}"
encodec_fn = "./pretrained_models/encodec_4cb2048_giga.th"

def download_file(url, dest):
    if not os.path.exists(dest):
        print(f"Downloading {os.path.basename(dest)}...")
        r = requests.get(url, stream=True)
        with open(dest, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print("Done")

os.makedirs("./pretrained_models", exist_ok=True)
download_file(f"https://huggingface.co/pyp1/VoiceCraft/resolve/main/{voicecraft_name}", ckpt_fn)
download_file("https://huggingface.co/pyp1/VoiceCraft/resolve/main/encodec_4cb2048_giga.th", encodec_fn)

ckpt = torch.load(ckpt_fn, map_location="cpu", weights_only=False)
model_args = ckpt["config"]
phn2num = ckpt["phn2num"]
model = voicecraft.VoiceCraft(model_args)
model.load_state_dict(ckpt["model"])
del ckpt; gc.collect()
model.to(device).train()
print(f"✅ Model loaded ({len(phn2num)} phonemes). GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")

# --- Hebrew Phonemizer ---
# espeak (used by VoiceCraft for English) doesn't support Hebrew.
# We map each Hebrew consonant to its IPA symbol. No vowels needed
# because Hebrew is normally written without vowel marks (niqqud).
def hebrew_phonemize(text):
    letter_map = {
        'א': 'ʔ', 'ב': 'v', 'ג': 'g', 'ד': 'd', 'ה': 'h',
        'ו': 'v', 'ז': 'z', 'ח': 'χ', 'ט': 't', 'י': 'j',
        'כ': 'χ', 'ך': 'χ', 'ל': 'l', 'מ': 'm', 'ם': 'm',
        'נ': 'n', 'ן': 'n', 'ס': 's', 'ע': 'ʕ', 'פ': 'f',
        'ף': 'f', 'צ': 'ts', 'ץ': 'ts', 'ק': 'k', 'ר': 'ʁ',
        'ש': 'ʃ', 'ת': 't',
    }
    result = []
    for word in text.split():
        phones = []
        for char in word:
            if char in letter_map:
                phones.append(letter_map[char])
        if phones:
            result.append(' '.join(phones))
    return ' '.join(result)

# 5 Hebrew phonemes don't exist in VoiceCraft's English vocabulary.
# We map them to the closest English equivalents. The model will learn
# the correct Hebrew sounds during finetuning.
HEBREW_TO_MODEL = {
    'g': 'ɡ',   # same sound, different unicode character
    'ts': 'tʃ', # both are t + fricative
    'ʁ': 'ɹ',   # both are R sounds (uvular vs alveolar)
    'χ': 'x',   # both are velar/uvular fricatives
    'ʕ': 'ɑ',   # pharyngeal approximated with open vowel
}

def hebrew_phonemize_mapped(text):
    raw = hebrew_phonemize(text)
    phones = raw.split()
    mapped = [HEBREW_TO_MODEL.get(p, p) for p in phones]
    return ' '.join(mapped)

# Verify all phonemes map to model vocabulary
test = hebrew_phonemize_mapped("שלום עולם")
all_ok = all(p in phn2num for p in test.split())
print(f"✅ Hebrew phonemizer: שלום עולם → {test} (all mapped: {all_ok})")

## Step 3: Prepare Hebrew Training Data\n\nDownloads Google FLEURS Hebrew dataset (3,242 samples), phonemizes all transcripts, and encodes audio with Encodec into VoiceCraft's training format.

In [ ]:
from datasets import load_dataset
from encodec.model import EncodecModel as EncodecModelPip
from encodec.quantization.vq import ResidualVectorQuantizer
from encodec.modules import SEANetEncoder, SEANetDecoder
import soundfile as sf
import librosa

# Load Encodec model (VoiceCraft's custom weights)
print("Loading Encodec...")
enc_ckpt = torch.load(encodec_fn, map_location="cpu", weights_only=False)
encoder = SEANetEncoder(channels=1, dimension=128, n_filters=64, n_residual_layers=1,
    ratios=[8,5,4,2], activation='ELU', norm='weight_norm', kernel_size=7,
    causal=False, pad_mode='constant', true_skip=True, compress=2, lstm=2)
decoder = SEANetDecoder(channels=1, dimension=128, n_filters=64, n_residual_layers=1,
    ratios=[8,5,4,2], activation='ELU', norm='weight_norm', kernel_size=7,
    causal=False, pad_mode='constant', true_skip=True, compress=2, lstm=2, trim_right_ratio=1.0)
quantizer = ResidualVectorQuantizer(dimension=128, n_q=4, bins=2048, decay=0.99,
    kmeans_init=True, kmeans_iters=50, threshold_ema_dead_code=2)
encodec_model = EncodecModelPip(encoder=encoder, decoder=decoder, quantizer=quantizer,
    target_bandwidths=[1.5,3.0,6.0,12.0], sample_rate=16000, channels=1)
encodec_model.load_state_dict(enc_ckpt['best_state']['model'])
encodec_model = encodec_model.to(device).eval()
del enc_ckpt; gc.collect()
print("✅ Encodec loaded")

# Download FLEURS Hebrew
print("\nDownloading FLEURS Hebrew dataset...")
ds = load_dataset("google/fleurs", "he_il", split="train", trust_remote_code=True)
ds_val = load_dataset("google/fleurs", "he_il", split="validation", trust_remote_code=True)
print(f"Downloaded: {len(ds)} train, {len(ds_val)} validation")

# Prepare directories
DATA_DIR = "/kaggle/working/training_data"
for d in ["phonemes", "codes", "encodec_16khz_4codebooks", "manifest"]:
    os.makedirs(f"{DATA_DIR}/{d}", exist_ok=True)

# Process all samples: phonemize text + encode audio
def process_split(dataset, split_name):
    entries = []
    skipped = 0
    for i in range(len(dataset)):
        sample = dataset[i]
        audio = np.array(sample["audio"]["array"], dtype=np.float32)
        sr = sample["audio"]["sampling_rate"]
        text = sample["transcription"]
        if sr != 16000:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
        duration = len(audio) / 16000
        if duration < 2.0 or duration > 16.0:
            skipped += 1; continue
        phones = hebrew_phonemize_mapped(text)
        if not phones.strip():
            skipped += 1; continue
        utt_id = f"{split_name}_{i:06d}"
        # Save phonemes
        with open(f"{DATA_DIR}/phonemes/{utt_id}.txt", "w") as f:
            f.write(phones)
        # Encode audio with Encodec
        audio_t = torch.FloatTensor(audio).unsqueeze(0).unsqueeze(0).to(device)
        with torch.no_grad():
            codes = encodec_model.encode(audio_t)[0][0].cpu()  # [1, 4, T]
        codes = codes.squeeze(0)  # [4, T]
        # Save codes as text (VoiceCraft format: one codebook per line)
        with open(f"{DATA_DIR}/encodec_16khz_4codebooks/{utt_id}.txt", "w") as f:
            for cb in range(codes.shape[0]):
                f.write(" ".join([str(int(c)) for c in codes[cb]]) + "\n")
        entries.append((utt_id, codes.shape[1]))
        if i % 300 == 0:
            print(f"  {split_name}: {i}/{len(dataset)}")
    print(f"  {split_name}: {len(entries)} kept, {skipped} skipped")
    return entries

print("\nProcessing training data...")
train_entries = process_split(ds, "train")
print("Processing validation data...")
val_entries = process_split(ds_val, "val")

# Write manifests in VoiceCraft's expected format (TSV: dummy \t utt_id \t n_frames)
for name, entries in [("train", train_entries), ("validation", val_entries)]:
    with open(f"{DATA_DIR}/manifest/{name}.txt", "w") as f:
        for utt_id, n_frames in entries:
            f.write(f"dummy\t{utt_id}\t{n_frames}\n")

# Write vocab
with open(f"{DATA_DIR}/vocab.txt", "w") as f:
    for phn, num in sorted(phn2num.items(), key=lambda x: x[1]):
        f.write(f"{num} {phn}\n")

# Free Encodec from GPU
encodec_model.cpu()
del encodec_model; gc.collect(); torch.cuda.empty_cache()

print(f"\n✅ Data ready! Train: {len(train_entries)}, Val: {len(val_entries)}, Vocab: {len(phn2num)}")

## Step 4: Finetune VoiceCraft on Hebrew\n\nTrains the pretrained English model on our Hebrew data. Uses VoiceCraft's own dataset class and forward pass (random masking + prediction).\n\nThis takes ~2-3 hours on Kaggle T4. Checkpoints are saved every 100 steps.

In [ ]:
from data.gigaspeech import dataset as VCDataset
from torch.optim import AdamW

EXP_DIR = "/kaggle/working/experiment"
os.makedirs(EXP_DIR, exist_ok=True)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

# Configure dataset loader
ds_args = argparse.Namespace(
    dataset_dir=DATA_DIR, exp_dir=EXP_DIR,
    manifest_name="manifest", phn_folder_name="phonemes",
    encodec_folder_name="encodec_16khz_4codebooks",
    n_codebooks=4, encodec_sr=50,
    audio_min_length=2, audio_max_length=8,
    text_max_length=200, text_min_length=10,
    drop_long=1, pad_x=0, dynamic_batching=0,
    text_pad_token=model_args.text_pad_token,
    audio_pad_token=model_args.audio_pad_token,
    special_first=0, sep_special_token=False, batch_size=1,
)

train_dataset = VCDataset(ds_args, "train")
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=1, shuffle=True,
    num_workers=0, collate_fn=train_dataset.collate, drop_last=True
)
print(f"Training samples: {len(train_dataset)}")

# Training config
EPOCHS = 10
GRAD_ACCUM = 8
LOG_EVERY = 20
SAVE_EVERY = 100  # save frequently to survive disconnections
LR = 1e-5

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')
global_step = 0

print(f"Epochs: {EPOCHS}, LR: {LR}, Effective batch: {GRAD_ACCUM}")
print(f"GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")
print("\nTraining started...\n")

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    n_batches = 0
    optimizer.zero_grad()

    for batch_idx, batch in enumerate(train_loader):
        try:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                out = model(batch)
                if out is None: continue
                loss = out['loss'] / GRAD_ACCUM
            scaler.scale(loss).backward()
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            if (batch_idx + 1) % GRAD_ACCUM == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                global_step += 1
                if global_step % LOG_EVERY == 0:
                    avg = epoch_loss / n_batches
                    print(f"  Epoch {epoch+1}, Step {global_step}, Loss: {avg:.2f}")
                if global_step % SAVE_EVERY == 0:
                    spath = f"/kaggle/working/checkpoints/step_{global_step}.pth"
                    torch.save({"model": model.state_dict(), "config": model_args,
                               "phn2num": phn2num, "step": global_step}, spath)
                    print(f"  💾 Saved: {spath}")
        except Exception as e:
            if "CUDA" in str(e): torch.cuda.empty_cache()
            continue

    avg_loss = epoch_loss / max(n_batches, 1)
    spath = f"/kaggle/working/checkpoints/epoch_{epoch+1}.pth"
    torch.save({"model": model.state_dict(), "config": model_args,
               "phn2num": phn2num, "step": global_step}, spath)
    print(f"\n✅ Epoch {epoch+1}/{EPOCHS} done. Loss: {avg_loss:.2f}. Saved: {spath}\n")

print(f"\n🎉 Training complete! {global_step} steps")

## Step 5: Test Hebrew Speech Editing\n\nLoad the finetuned model and edit a word in a Hebrew audio clip.

In [ ]:
# Load best checkpoint
import glob
ckpt_files = sorted(glob.glob("/kaggle/working/checkpoints/epoch_*.pth"))
if ckpt_files:
    best_ckpt = ckpt_files[-1]
else:
    best_ckpt = sorted(glob.glob("/kaggle/working/checkpoints/step_*.pth"))[-1]

print(f"Loading checkpoint: {best_ckpt}")
ckpt = torch.load(best_ckpt, map_location="cpu", weights_only=False)
model = voicecraft.VoiceCraft(ckpt["config"])
model.load_state_dict(ckpt["model"])
model.to(device).eval()
phn2num = ckpt["phn2num"]
model_args = ckpt["config"]
del ckpt; gc.collect()

# Load a Hebrew test sample
from datasets import load_dataset
from IPython.display import Audio, display

ds_test = load_dataset("google/fleurs", "he_il", split="test", trust_remote_code=True)
sample = ds_test[0]
audio = np.array(sample["audio"]["array"], dtype=np.float32)
sr = sample["audio"]["sampling_rate"]
if sr != 16000:
    audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)

orig_transcript = sample["transcription"]
print(f"Original: {orig_transcript}")
print("\nOriginal audio:")
display(Audio(audio, rate=16000))

# Load Encodec for decoding
enc_ckpt = torch.load(encodec_fn, map_location="cpu", weights_only=False)
encoder = SEANetEncoder(channels=1, dimension=128, n_filters=64, n_residual_layers=1,
    ratios=[8,5,4,2], activation='ELU', norm='weight_norm', kernel_size=7,
    causal=False, pad_mode='constant', true_skip=True, compress=2, lstm=2)
decoder = SEANetDecoder(channels=1, dimension=128, n_filters=64, n_residual_layers=1,
    ratios=[8,5,4,2], activation='ELU', norm='weight_norm', kernel_size=7,
    causal=False, pad_mode='constant', true_skip=True, compress=2, lstm=2, trim_right_ratio=1.0)
quantizer = ResidualVectorQuantizer(dimension=128, n_q=4, bins=2048, decay=0.99,
    kmeans_init=True, kmeans_iters=50, threshold_ema_dead_code=2)
encodec_model = EncodecModelPip(encoder=encoder, decoder=decoder, quantizer=quantizer,
    target_bandwidths=[1.5,3.0,6.0,12.0], sample_rate=16000, channels=1)
encodec_model.load_state_dict(enc_ckpt['best_state']['model'])
encodec_model = encodec_model.to(device).eval()
del enc_ckpt

# Encode audio
audio_t = torch.FloatTensor(audio).unsqueeze(0).unsqueeze(0).to(device)
with torch.no_grad():
    original_codes = encodec_model.encode(audio_t)[0][0]  # [1, 4, T]

# Pick a word to edit
words = orig_transcript.split()
print(f"\nWords:")
for i, w in enumerate(words):
    print(f"  {i}: {w}")

# Change word (modify these two lines to edit different words)
EDIT_IDX = 0  # which word to change
NEW_WORD = "אתמול"  # what to change it to

target_words = words.copy()
target_words[EDIT_IDX] = NEW_WORD
target_transcript = " ".join(target_words)

print(f"\nChanging: {words[EDIT_IDX]} → {NEW_WORD}")
print(f"Target: {target_transcript}")

# Phonemize and tokenize
target_phones = hebrew_phonemize_mapped(target_transcript)
phone_list = target_phones.split()
text_tokens = torch.LongTensor([phn2num[p] for p in phone_list if p in phn2num]).unsqueeze(0).to(device)
text_tokens_lens = torch.LongTensor([text_tokens.shape[-1]]).to(device)
original_audio = original_codes.transpose(1, 2)  # [1, T, 4]

# Create mask (evenly spaced alignment)
duration = len(audio) / 16000
word_dur = duration / len(words)
start = EDIT_IDX * word_dur
end = (EDIT_IDX + 1) * word_dur
mask_start = max(round((start - 0.08) * 50), 1)
mask_end = min(round((end + 0.08) * 50), original_audio.shape[1])
mask_interval = torch.LongTensor([[mask_start, mask_end]])

# Generate
print("Generating...")
with torch.no_grad():
    encoded_frames = model.inference(
        text_tokens, text_tokens_lens,
        original_audio[...,:model_args.n_codebooks].to(device),
        mask_interval=mask_interval.unsqueeze(0).to(device),
        top_k=0, top_p=0.8, temperature=1,
        stop_repetition=-1, kvcache=0,
        silence_tokens=[1388, 1898, 131],
    )
if isinstance(encoded_frames, tuple):
    encoded_frames = encoded_frames[0]

with torch.no_grad():
    original_sample = encodec_model.decode([(original_codes, None)])
    generated_sample = encodec_model.decode([(encoded_frames, None)])

print("\nOriginal (re-encoded):")
display(Audio(original_sample[0].squeeze().cpu().numpy(), rate=16000))
print(f"\nEdited ({words[EDIT_IDX]} → {NEW_WORD}):")
display(Audio(generated_sample[0].squeeze().cpu().numpy(), rate=16000))
print("\n✅ Hebrew speech editing complete!")